# Create National Psoriasis Foundation Awards

Builds the awards table for the **National Psoriasis Foundation** (funder_id
`4320307379`, IAMHRF expanded list, priority `293`) from psoriasis.org's
funded-research-projects accordion + detail pages. 49 active projects with PI
(44/49), institution, USD amount (48/49), scheme, and project start/end dates.

Source parquet: `s3://openalex-ingest/awards/npf/npf_grants.parquet`
(produced by `scripts/local/npf_to_s3.py`).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.npf_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/npf/npf_grants.parquet`;

In [ ]:
%sql
-- Check row count (should be 49)
SELECT COUNT(*) as total_projects FROM openalex.awards.npf_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.npf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.npf_raw LIMIT 5;

## Step 2: Create NPF Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.npf_awards
USING delta
AS
WITH
npf_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320307379  -- National Psoriasis Foundation
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        CAST(NULL AS STRING) as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        TRY_CAST(g.amount AS DECIMAL(18,2)) as amount,
        CASE WHEN g.amount IS NOT NULL THEN 'USD' END as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        g.scheme as funder_scheme,
        'npf' as provenance,
        TRY_TO_DATE(g.start_date_raw, 'MMMM d, yyyy') as start_date,
        TRY_TO_DATE(g.end_date_raw, 'MMMM d, yyyy') as end_date,
        YEAR(TRY_TO_DATE(g.start_date_raw, 'MMMM d, yyyy')) as start_year,
        YEAR(TRY_TO_DATE(g.end_date_raw, 'MMMM d, yyyy')) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(
                    g.pi_given as given_name,
                    g.pi_family as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        CAST(NULL AS STRING) as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.npf_raw g
    CROSS JOIN npf_funder f
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'npf' AND priority = 293;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    293 as priority
FROM openalex.awards.npf_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards FROM openalex.awards.npf_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, amount, currency, start_year, end_year, funder_scheme
FROM openalex.awards.npf_awards LIMIT 10;

In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.npf_awards GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount)/1000, 0) as funding_k_usd
FROM openalex.awards.npf_awards GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(lead_investigator) as has_pi,
    COUNT(start_date) as has_start_date,
    ROUND(SUM(amount)/1000000, 2) as total_funding_million_usd
FROM openalex.awards.npf_awards;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name as institution, COUNT(*) as cnt
FROM openalex.awards.npf_awards
WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY 1 ORDER BY 2 DESC LIMIT 15;

In [ ]:
%sql
SELECT COUNT(*) as npf_in_raw
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'npf' AND priority = 293;